In [3]:
# ============================================================
# Grouped EDA (with numeric annotations on target + corr)
# Data: ./training_data.csv
# Output: ./eda_out_grouped/
# Also saves normalized CSV: ./training_data_normalized.csv
#
# Policy:
# - If "Unnamed: 0" exists, NEVER transform it (no cast / no normalization / exclude from EDA feature lists).
# - Else, NEVER transform the first column.
# ============================================================

import os
import math
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# ----------------------------
# 0) Config
# ----------------------------
DATA_PATH = "./training_data.csv"
OUT_DIR = "./eda_out_grouped"
RANDOM_STATE = 42

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

MAX_VARS_DIST = 24      # hist grid에 넣을 수치변수 개수
MAX_VARS_BOX = 24       # box grid에 넣을 수치변수 개수
MAX_VARS_CORR = 35      # corr heatmap에 넣을 수치변수 개수 (annotate는 35 이하에서만)
GRID_NCOLS = 4          # grid 열 개수

# --- Normalization options ---
APPLY_NORMALIZATION = True
NORMALIZED_CSV_PATH = "./training_data_normalized.csv"
# ----------------------------

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figs"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "tables"), exist_ok=True)

# ----------------------------
# 1) Utils
# ----------------------------
def save_fig(fig, name: str):
    path = os.path.join(OUT_DIR, "figs", name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def try_cast_numeric(df: pd.DataFrame, skip_cols=None):
    """
    object 컬럼 중 대부분이 숫자 문자열이면 numeric으로 변환 시도
    - skip_cols: 변환 금지할 컬럼 이름들
    """
    if skip_cols is None:
        skip_cols = set()
    else:
        skip_cols = set(skip_cols)

    df2 = df.copy()
    for c in df2.columns:
        if c in skip_cols:
            continue
        if df2[c].dtype == "object":
            s = df2[c].dropna().astype(str)
            if len(s) == 0:
                continue
            sample = s.head(300)
            numeric_like = sample.str.match(r"^\s*-?\d+(\.\d+)?\s*$").mean()
            if numeric_like > 0.85:
                df2[c] = pd.to_numeric(df2[c], errors="coerce")
    return df2

def infer_target_col(df: pd.DataFrame):
    # 1) 이름 후보 우선
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c

    # 2) 이진 컬럼 탐색
    best = None
    for c in df.columns:
        s = df[c].dropna()
        if s.empty:
            continue
        uniq = pd.unique(s)
        if len(uniq) == 2:
            # 0/1 숫자이면 우선
            if pd.api.types.is_numeric_dtype(df[c]):
                u = sorted(pd.unique(s))
                if set(u).issubset({0, 1}):
                    return c
            if best is None:
                best = c
    return best

def summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n = len(df)
    for c in df.columns:
        s = df[c]
        rows.append([
            c,
            str(s.dtype),
            int(s.nunique(dropna=True)),
            float(s.isna().mean()),
            int(n),
        ])
    out = pd.DataFrame(rows, columns=["column", "dtype", "nunique", "missing_ratio", "n_rows"])
    out = out.sort_values(["missing_ratio", "nunique"], ascending=[False, False]).reset_index(drop=True)
    return out

def numeric_summary(df: pd.DataFrame, num_cols: list) -> pd.DataFrame:
    if not num_cols:
        return pd.DataFrame()
    desc = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    desc["missing_ratio"] = df[num_cols].isna().mean()
    desc["skew"] = df[num_cols].skew(numeric_only=True)
    desc["kurtosis"] = df[num_cols].kurtosis(numeric_only=True)
    return desc.reset_index().rename(columns={"index": "column"})

def categorical_topk_tables(df: pd.DataFrame, cat_cols: list, topk=20):
    for c in cat_cols:
        vc = df[c].astype("object").value_counts(dropna=False).head(topk)
        out = vc.to_frame("count").reset_index().rename(columns={"index": c})
        safe_c = c.replace("/", "_").replace("\\", "_")
        out.to_csv(os.path.join(OUT_DIR, "tables", f"cat_top_{safe_c}.csv"), index=False)

def select_numeric_cols_for_plots(df: pd.DataFrame, num_cols: list, max_vars: int):
    """
    (1) 결측률 낮음
    (2) 표준편차 > 0
    기준으로 상위 max_vars 선택
    """
    if not num_cols:
        return []
    stats = []
    for c in num_cols:
        s = df[c]
        miss = s.isna().mean()
        try:
            std = float(s.dropna().std(ddof=1))
        except Exception:
            std = 0.0
        stats.append((c, miss, std))
    tmp = pd.DataFrame(stats, columns=["column", "missing_ratio", "std"])
    tmp = tmp[tmp["std"].fillna(0) > 0].copy()
    tmp = tmp.sort_values(["missing_ratio", "std"], ascending=[True, False])
    return tmp["column"].head(max_vars).tolist()

def normalize_numeric_columns(df: pd.DataFrame, numeric_cols: list):
    """
    수치 컬럼만 Z-score 정규화(평균0, 표준편차1).
    - 결측치는 유지(NaN 그대로)
    - std=0(상수) 컬럼은 스킵
    반환: (정규화된 df, stats_df)
    """
    df_out = df.copy()
    stats = []

    for c in numeric_cols:
        s = df_out[c]
        s_nonan = s.dropna()
        if s_nonan.empty:
            stats.append([c, np.nan, np.nan, "skip_empty"])
            continue

        mean = float(s_nonan.mean())
        std = float(s_nonan.std(ddof=0))  # population std

        if std == 0 or np.isnan(std):
            stats.append([c, mean, std, "skip_std0"])
            continue

        df_out[c] = (s - mean) / std
        stats.append([c, mean, std, "scaled"])

    stats_df = pd.DataFrame(stats, columns=["column", "mean_used", "std_used", "status"])
    return df_out, stats_df

# ----------------------------
# 2) Plotters
# ----------------------------
def plot_missing_ratio_bar(col_summary: pd.DataFrame, topn=50):
    tmp = col_summary.sort_values("missing_ratio", ascending=False).head(topn)
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111)
    ax.barh(tmp["column"][::-1], tmp["missing_ratio"][::-1])
    ax.set_title(f"Missing ratio (top {topn})")
    ax.set_xlabel("missing_ratio")
    fig.tight_layout()
    save_fig(fig, f"missing_ratio_top{topn}.png")

def plot_target_distribution(y: pd.Series, target_col: str):
    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111)

    vc = y.value_counts(dropna=False)
    total = vc.sum()

    bars = ax.bar(vc.index.astype(str), vc.values)
    ax.set_title(f"Target distribution: {target_col}")
    ax.set_xlabel("class")
    ax.set_ylabel("count")

    for b, v in zip(bars, vc.values):
        pct = (v / total) * 100 if total > 0 else 0
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height(),
            f"{int(v)}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=10
        )

    fig.tight_layout()
    save_fig(fig, "target_distribution.png")

def plot_hist_grid(df: pd.DataFrame, cols: list, n_cols=4, bins=40,
                   title="Numeric Feature Distributions", out_name="hist_grid.png"):
    cols = [c for c in cols if c in df.columns]
    if not cols:
        return

    n = len(cols)
    n_rows = math.ceil(n / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    axes = np.array(axes).reshape(-1)

    for i, c in enumerate(cols):
        s = df[c].dropna()
        axes[i].hist(s.values, bins=bins)
        axes[i].set_title(c, fontsize=10)
        axes[i].tick_params(axis="both", labelsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    save_fig(fig, out_name)

def plot_box_by_target_grid(df: pd.DataFrame, cols: list, target_col: str, n_cols=4,
                            title="Numeric Features by Target", out_name="box_by_target_grid.png"):
    cols = [c for c in cols if c in df.columns and c != target_col]
    if not cols:
        return

    classes = sorted(df[target_col].dropna().unique())
    if len(classes) < 2:
        return

    n = len(cols)
    n_rows = math.ceil(n / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    axes = np.array(axes).reshape(-1)

    for i, c in enumerate(cols):
        data = [df.loc[df[target_col] == cls, c].dropna().values for cls in classes]
        axes[i].boxplot(data, labels=[str(cls) for cls in classes], showfliers=False)
        axes[i].set_title(c, fontsize=10)
        axes[i].tick_params(axis="both", labelsize=8)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    save_fig(fig, out_name)

def plot_corr_heatmap(df: pd.DataFrame, cols: list,
                      title="Correlation Heatmap (Numeric Features)",
                      out_name="corr_heatmap.png",
                      annotate=True, fmt="{:.2f}"):
    cols = [c for c in cols if c in df.columns]
    if len(cols) < 2:
        return

    corr = df[cols].corr(numeric_only=True)
    corr.to_csv(os.path.join(OUT_DIR, "tables", "corr_matrix.csv"))

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(corr.values, aspect="auto", vmin=-1, vmax=1)

    ax.set_xticks(range(len(cols)))
    ax.set_yticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=90, fontsize=8)
    ax.set_yticklabels(cols, fontsize=8)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)

    if annotate:
        if len(cols) <= 35:
            for i in range(len(cols)):
                for j in range(len(cols)):
                    val = corr.values[i, j]
                    ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=6)
        else:
            print(f"[WARN] Too many cols ({len(cols)}) -> skip annotation to avoid clutter.")

    fig.tight_layout()
    save_fig(fig, out_name)

# ----------------------------
# 3) Target association tables
# ----------------------------
def corr_with_target_numeric(df: pd.DataFrame, num_cols: list, target_col: str) -> pd.DataFrame:
    y = pd.to_numeric(df[target_col], errors="coerce")
    out = []
    for c in num_cols:
        tmp = pd.concat([df[c], y.rename(target_col)], axis=1).dropna()
        if tmp.empty:
            continue
        if tmp[target_col].nunique() < 2:
            continue
        corr = tmp[c].corr(tmp[target_col])
        out.append([c, corr, len(tmp)])
    res = pd.DataFrame(out, columns=["column", "corr_with_target", "n_used"])
    if not res.empty:
        res = res.sort_values("corr_with_target", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
    return res

def mean_diffs_by_target(df: pd.DataFrame, num_cols: list, target_col: str) -> pd.DataFrame:
    tmp = df[[target_col] + num_cols].copy()
    tmp = tmp.dropna(subset=[target_col])
    classes = sorted(tmp[target_col].dropna().unique())
    if len(classes) != 2:
        return pd.DataFrame()

    a, b = classes[0], classes[1]
    res = []
    for c in num_cols:
        x0 = tmp.loc[tmp[target_col] == a, c].dropna()
        x1 = tmp.loc[tmp[target_col] == b, c].dropna()
        if len(x0) < 10 or len(x1) < 10:
            continue
        m0, m1 = x0.mean(), x1.mean()
        s0, s1 = x0.std(ddof=1), x1.std(ddof=1)
        denom = (len(x0) + len(x1) - 2)
        sp = math.sqrt(((len(x0)-1)*s0*s0 + (len(x1)-1)*s1*s1) / denom) if denom > 0 else np.nan
        d = (m1 - m0) / sp if sp and not np.isnan(sp) and sp > 0 else np.nan
        res.append([c, len(x0), len(x1), m0, m1, (m1 - m0), d])

    out = pd.DataFrame(res, columns=["column", "n0", "n1", "mean0", "mean1", "mean_diff(1-0)", "cohen_d(approx)"])
    if not out.empty:
        out = out.sort_values("cohen_d(approx)", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
    return out

# ----------------------------
# 4) Baseline sanity check
# ----------------------------
def baseline_logreg(df: pd.DataFrame, target_col: str):
    y = df[target_col].dropna()
    if y.nunique() != 2:
        return None

    X = df.drop(columns=[target_col])
    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]

    TOPK = 60
    X2 = X.copy()
    for c in cat_cols:
        vc = X2[c].value_counts(dropna=False)
        keep = set(vc.head(TOPK).index)
        X2[c] = X2[c].where(X2[c].isin(keep), other="__OTHER__")

    data = pd.concat([X2, df[target_col]], axis=1).dropna(subset=[target_col])
    X2 = data.drop(columns=[target_col])
    y2 = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X2, y2, test_size=0.25, random_state=RANDOM_STATE, stratify=y2
    )

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]), cat_cols),
        ],
        remainder="drop",
    )

    clf = LogisticRegression(max_iter=2000)
    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(X_train, y_train)

    p = pipe.predict_proba(X_test)[:, 1]
    try:
        auc = roc_auc_score(y_test, p)
        ap = average_precision_score(y_test, p)
    except Exception:
        auc, ap = None, None

    return {
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "num_cols": int(len(num_cols)),
        "cat_cols": int(len(cat_cols)),
        "roc_auc": auc,
        "avg_precision": ap
    }

# ----------------------------
# 5) Run
# ----------------------------
print(f"[LOAD] {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("[INFO] raw shape:", df_raw.shape)

# Decide skip column:
# - Prefer "Unnamed: 0" if exists
# - Else skip the first column
if "Unnamed: 0" in df_raw.columns:
    skip_col = "Unnamed: 0"
else:
    skip_col = df_raw.columns[0]

SKIP_COLS = {skip_col}
print(f"[INFO] skip_col (no cast / no normalize / exclude from feature lists): {skip_col}")

# cast numeric (but not skip_col)
df = try_cast_numeric(df_raw, skip_cols=SKIP_COLS)

target_col = infer_target_col(df)
if target_col is None:
    raise ValueError(
        "타깃 컬럼을 자동으로 찾지 못했습니다. "
        "TARGET_CANDIDATES에 이름을 추가하거나 target_col을 수동 지정하세요."
    )
print("[INFO] target_col:", target_col)

# ---- Normalization (only numeric cols except target + skip col) ----
X_cols = [c for c in df.columns if c != target_col and c not in SKIP_COLS]
num_cols_for_norm = [c for c in X_cols if pd.api.types.is_numeric_dtype(df[c])]

if APPLY_NORMALIZATION and len(num_cols_for_norm) > 0:
    df, norm_stats = normalize_numeric_columns(df, num_cols_for_norm)
    norm_stats.to_csv(os.path.join(OUT_DIR, "tables", "normalization_stats.csv"), index=False)

    df.to_csv(NORMALIZED_CSV_PATH, index=False)
    print(f"[INFO] normalized CSV saved: {os.path.abspath(NORMALIZED_CSV_PATH)}")
    print(f"[INFO] normalization applied to numeric cols: {int((norm_stats['status']=='scaled').sum())} scaled")
else:
    print("[INFO] normalization skipped (no numeric cols or APPLY_NORMALIZATION=False)")

# summaries (include all columns; skip_col will appear as-is)
col_sum = summarize_df(df)
col_sum.to_csv(os.path.join(OUT_DIR, "tables", "column_summary.csv"), index=False)

# feature groups for downstream tables/plots (exclude target + skip col)
X_cols = [c for c in df.columns if c != target_col and c not in SKIP_COLS]
num_cols_all = [c for c in X_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols_all = [c for c in X_cols if c not in num_cols_all]

print("[INFO] numeric cols (excluding target/skip):", len(num_cols_all))
print("[INFO] categorical cols (excluding target/skip):", len(cat_cols_all))

num_sum = numeric_summary(df, num_cols_all)
if not num_sum.empty:
    num_sum.to_csv(os.path.join(OUT_DIR, "tables", "numeric_summary.csv"), index=False)

categorical_topk_tables(df, cat_cols_all, topk=20)

meta = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "data_path": DATA_PATH,
    "raw_shape": [int(df_raw.shape[0]), int(df_raw.shape[1])],
    "shape_after_cast_and_norm": [int(df.shape[0]), int(df.shape[1])],
    "target_col": target_col,
    "skip_col": skip_col,
    "numeric_cols_used": int(len(num_cols_all)),
    "categorical_cols_used": int(len(cat_cols_all)),
    "apply_normalization": bool(APPLY_NORMALIZATION),
    "normalized_csv_path": NORMALIZED_CSV_PATH if APPLY_NORMALIZATION else None
}
with open(os.path.join(OUT_DIR, "tables", "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# plots
plot_missing_ratio_bar(col_sum, topn=50)
plot_target_distribution(df[target_col], target_col)

num_cols_dist = select_numeric_cols_for_plots(df, num_cols_all, MAX_VARS_DIST)
num_cols_box = select_numeric_cols_for_plots(df, num_cols_all, MAX_VARS_BOX)
num_cols_corr = select_numeric_cols_for_plots(df, num_cols_all, MAX_VARS_CORR)

plot_hist_grid(
    df, num_cols_dist,
    n_cols=GRID_NCOLS,
    bins=40,
    title=f"Numeric Feature Distributions (top {len(num_cols_dist)})",
    out_name="hist_grid.png"
)

plot_box_by_target_grid(
    df, num_cols_box,
    target_col=target_col,
    n_cols=GRID_NCOLS,
    title=f"Numeric Features by Target (top {len(num_cols_box)})",
    out_name="box_by_target_grid.png"
)

plot_corr_heatmap(
    df, num_cols_corr,
    title=f"Correlation Heatmap (top {len(num_cols_corr)} numeric cols)",
    out_name="corr_heatmap.png",
    annotate=True,
    fmt="{:.2f}"
)

# target association tables
pb = corr_with_target_numeric(df, num_cols_all, target_col)
if not pb.empty:
    pb.to_csv(os.path.join(OUT_DIR, "tables", "corr_with_target_numeric.csv"), index=False)

diffs = mean_diffs_by_target(df, num_cols_all, target_col)
if not diffs.empty:
    diffs.to_csv(os.path.join(OUT_DIR, "tables", "mean_diffs_by_target.csv"), index=False)

# leakage heuristic (on X_cols only)
leak_flags = []
if not pb.empty:
    for _, row in pb.head(30).iterrows():
        if pd.notna(row["corr_with_target"]) and abs(row["corr_with_target"]) > 0.95:
            leak_flags.append({"column": row["column"], "reason": "abs(corr_with_target) > 0.95"})

sus_keywords = ["label", "target", "fail", "default", "bankrupt", "bankruptcy", "outcome"]
for c in X_cols:
    low = c.lower()
    if any(k in low for k in sus_keywords):
        leak_flags.append({"column": c, "reason": "suspicious_keyword_in_column_name"})

pd.DataFrame(leak_flags).drop_duplicates().to_csv(
    os.path.join(OUT_DIR, "tables", "leakage_flags.csv"), index=False
)

# baseline sanity check (include skip col? usually no; using df without target only, so skip col is in X unless excluded)
# Here: we run baseline on df where skip col is included as a feature ONLY if it is not in SKIP_COLS, but it is in SKIP_COLS.
# So baseline will NOT see skip col, consistent with EDA feature lists.
df_for_baseline = df.drop(columns=list(SKIP_COLS), errors="ignore")

base = baseline_logreg(df_for_baseline, target_col)
if base is not None:
    with open(os.path.join(OUT_DIR, "tables", "baseline_logreg.json"), "w", encoding="utf-8") as f:
        json.dump(base, f, ensure_ascii=False, indent=2)
    print("[BASELINE]", base)
else:
    print("[BASELINE] skipped (target not binary?)")

print("[DONE] Saved outputs to:", os.path.abspath(OUT_DIR))
print("[OUTPUTS]")
print(" - tables :", os.path.join(os.path.abspath(OUT_DIR), "tables"))
print(" - figs   :", os.path.join(os.path.abspath(OUT_DIR), "figs"))
print(" - normalized csv :", os.path.abspath(NORMALIZED_CSV_PATH))
print(" - skip_col preserved:", skip_col)


[LOAD] ./training_data.csv
[INFO] raw shape: (17881, 15)
[INFO] skip_col (no cast / no normalize / exclude from feature lists): Unnamed: 0
[INFO] target_col: label
[INFO] normalized CSV saved: d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\2_data_normalization\training_data_normalized.csv
[INFO] normalization applied to numeric cols: 13 scaled
[INFO] numeric cols (excluding target/skip): 13
[INFO] categorical cols (excluding target/skip): 0
[BASELINE] {'n_train': 13410, 'n_test': 4471, 'num_cols': 13, 'cat_cols': 0, 'roc_auc': np.float64(0.8908157512915063), 'avg_precision': np.float64(0.33702396703213616)}
[DONE] Saved outputs to: d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\2_data_normalization\eda_out_grouped
[OUTPUTS]
 - tables : d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\2_data_normalization\eda_out_grouped\tables
 - figs   : d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\2_data_normalization\eda_out_grouped\figs
 - normalized csv : d:\Univers